# Customer Churn Prediction Notebook

Welcome! This notebook is the required starting point and submission format for the AI task.

## How this notebook works

- Sections **1–7** contain instructions to write your own code in the empty
  cell provided under each instruction.
- Section **8** is pre-written for evaluator use and **must not be modified**. We will run
  it against a **hidden test set** to score your submission.
- Your notebook must run **top-to-bottom without errors** before submission
  (`Restart Kernel & Run All`).

## 0. Setup

Standard imports, and file paths are set up below.
Add any additional imports your approach needs in the same cell.


In [4]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)

# File paths
TRAIN_PATH = "/data/dataset-1.csv"        # <-- set this to your local copy of the training CSV
TARGET_COLUMN = "Churn"    # <-- set this to the churn label column name
EVALUATION_SPLIT_PATH = "./evaluation_split.csv"  # organizers only — do not rely on this file existing locally

## 1. Load the Data

Load the dataset from `TRAIN_PATH` into a dataframe.

In [5]:
df = pd.read_csv(TRAIN_PATH)
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,4950-BDEUX,Male,0,No,No,35,No,No phone service,DSL,No,...,Yes,No,Yes,Yes,Month-to-month,No,Electronic check,49.20,1701.65,False
1,7993-NQLJE,Male,0,Yes,Yes,15,Yes,No,Fiber optic,Yes,...,No,No,No,No,Month-to-month,No,Mailed check,75.10,1151.55,False
2,7321-ZNSLA,Male,0,Yes,Yes,13,No,No phone service,DSL,Yes,...,No,Yes,No,No,Two year,No,Mailed check,40.55,590.35,False
3,4922-CVPDX,Female,0,Yes,No,26,Yes,No,DSL,No,...,Yes,No,Yes,Yes,Two year,Yes,Credit card (automatic),73.50,1905.7,False
4,2903-YYTBW,Male,0,Yes,Yes,1,Yes,No,DSL,No,...,No,No,No,No,Month-to-month,No,Electronic check,44.55,44.55,False


## 2. Explore and Clean the Dataset

Address data quality issues before modeling:

- Identify and handle missing / null values.
- Identify categorical variables and decide how each will be handled.
- Check for duplicate rows, inconsistent categories, and outliers.
- Document any rows/columns you drop and why.

In [11]:
def data_cleaning(df):

    # Implement your cleaning steps here

    df = df.copy()

    if "customerID" in df.columns:
        df = df.drop(columns=["customerID"])


    if "TotalCharges" in df.columns:
        df["TotalCharges"] = pd.to_numeric(
            df["TotalCharges"],
            errors="coerce"
        )

    return df

## 3. Feature Engineering

Transform the raw columns into features that are more useful for a classification model.

For example: combining related columns, extracting information from
dates or creating new columns from existing ones.

In [13]:
def feature_engineering(df):
    # Implement your feature engineering steps here

    df = df.copy();

    if "tenure" in df.columns:
        df["IsNewCustomer"] = (df["tenure"] <= 6).astype(int)

    # Estimate charges
    if "MonthlyCharges" in df.columns and "tenure" in df.columns:
        df["EstimatedCharges"] = df["MonthlyCharges"] * df["tenure"]

    return df



## 4. Transform, Normalize, and Scale Features

Prepare the final feature set for modeling:

- Encode categorical variables.
- Scale/normalize numeric features as appropriate for the model(s) you plan to use.

In [14]:
def scale_features(df):
    # Implement your preprocessing steps here
    # This function should return a cleaned and feature-engineered dataframe
    return df.copy()

In [15]:
# Do not modify this function
def preprocess_data(df):
    df = df.copy()
    df = data_cleaning(df)
    df = feature_engineering(df)
    df = scale_features(df)

    return df

## 5. Train / Validation / Test Split

Split your cleaned data into three sets:

- **Train** — used to fit models.
- **Validation** — used for hyperparameter tuning and model comparison.
- **Test** — held out until Section 6.

In [16]:
def split_data(df):
    # Implement your data splitting logic here
    # This function should return three dataframes: train, validation, and test

    train_split, temp_split = train_test_split(
        df,
        test_size=0.30,
        random_state=42,
        stratify=df[TARGET_COLUMN]
    )

    validation_split, test_split = train_test_split(
        temp_split,
        test_size=0.50,
        random_state=42,
        stratify=temp_split[TARGET_COLUMN]
    )

    return train_split, validation_split, test_split

## 6. Model Training, and Hyperparameter Tuning

- Train classification models on the train set (try more than one algorithm and compare).
- Tune hyperparameters using the validation set.
- Use the `evaluate_predictions()` function (defined in Section 0) to report **train** and **validation**
  metrics for each model: Accuracy, Precision, Recall, F1-score, ROC-AUC.
- Select your best-performing model/configuration.

### Required output of this section
- **`predict_churn(df)`** — takes a raw dataframe, calls `preprocess_data(df)` internally, then returns `(predictions, probabilities)` from your best trained model.

This is what Section 8 will call on the hidden test set, so both functions must work on data that hasn't been touched yet — don't rely on variables that only exist because you already cleaned `df` earlier in the notebook.

In [22]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

# Split the RAW dataset first
train_split, validation_split, test_split = split_data(df)

# Separate features and target
X_train_raw = train_split.drop(columns=[TARGET_COLUMN])
y_train = train_split[TARGET_COLUMN].astype(int)

X_val_raw = validation_split.drop(columns=[TARGET_COLUMN])
y_val = validation_split[TARGET_COLUMN].astype(int)

X_test_raw = test_split.drop(columns=[TARGET_COLUMN])
y_test = test_split[TARGET_COLUMN].astype(int)


# Apply our cleaning + feature engineering
X_train = preprocess_data(X_train_raw)
X_val = preprocess_data(X_val_raw)
X_test = preprocess_data(X_test_raw)


# Find categorical and numerical columns
categorical_cols = X_train.select_dtypes(
    include=["object", "category"]
).columns.tolist()

numerical_cols = X_train.select_dtypes(
    include=["number"]
).columns.tolist()


# Preprocessing for numerical columns
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])


# Preprocessing for categorical columns
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])


# Combine preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numerical_cols),
        ("cat", categorical_transformer, categorical_cols)
    ]
)


# Logistic Regression model
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])


# Train ONLY using training data
model.fit(X_train, y_train)



train_predictions = model.predict(X_train)
train_probabilities = model.predict_proba(X_train)[:, 1]

print("TRAIN RESULTS")
evaluate_predictions(
    y_train,
    train_predictions,
    train_probabilities
)



val_probabilities = model.predict_proba(X_val)[:, 1]

# Start with normal threshold
threshold = 0.5
val_predictions = (val_probabilities >= threshold).astype(int)

print("\nVALIDATION RESULTS")
evaluate_predictions(
    y_val,
    val_predictions,
    val_probabilities
)


def predict_churn(df):

    processed = preprocess_data(df)
    probabilities = model.predict_proba(processed)[:, 1]
    predictions = (probabilities >= threshold).astype(int)

    return predictions, probabilities


TRAIN RESULTS
--- Metrics ---
  accuracy: 0.7593
 precision: 0.5303
    recall: 0.8117
  f1_score: 0.6415
   roc_auc: 0.8546

VALIDATION RESULTS
--- Metrics ---
  accuracy: 0.7692
 precision: 0.5443
    recall: 0.7946
  f1_score: 0.6461
   roc_auc: 0.8445


## 7. Evaluate on the Held-Out Test Set

Run `predict_churn()` on your held-out test split (never used in training or tuning) and report
the same metrics using `evaluate_predictions()`.

In [20]:
def evaluate_predictions(y_true, y_pred, y_proba):
    # DO NOT modify
    # Standard metric report used throughout this notebook.
    # Both your own evaluation (Section 6) and the grading step (Section 8) use this function.
    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1_score": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_proba),
    }
    print("--- Metrics ---")
    for name, value in metrics.items():
        print(f"{name:>10}: {value:.4f}")
    return metrics

In [23]:
# Predict on the held-out test set
test_predictions, test_probabilities = predict_churn(X_test_raw)

print("TEST RESULTS")

test_metrics = evaluate_predictions(
    y_test,
    test_predictions,
    test_probabilities
)

TEST RESULTS
--- Metrics ---
  accuracy: 0.7541
 precision: 0.5263
    recall: 0.7556
  f1_score: 0.6204
   roc_auc: 0.8435


---
## 8. Standardized Evaluation — DO NOT MODIFY

Everything below is run by the organizers for every submission. It loads the hidden evaluation set
and scores your submission. Make sure your notebook runs top-to-bottom.


In [ ]:
# =====================================================================
# ORGANIZER EVALUATION CELL — DO NOT MODIFY
# =====================================================================
assert "predict_churn" in dir(), (
    "predict_churn(df) is not defined. Your submission must define this "
    "function in Section 6 (see instructions there)."
)

evaluation_df = pd.read_csv(EVALUATION_SPLIT_PATH)
X_hidden = evaluation_df.drop(columns=[TARGET_COLUMN])
y_hidden = evaluation_df[TARGET_COLUMN]

y_pred, y_proba = predict_churn(X_hidden)

evaluation_metrics = evaluate_predictions(y_hidden, y_pred, y_proba)
pd.DataFrame([evaluation_metrics])